In [113]:
from classifiers import load_split, build_model, train, predict
from tokenization import compare_tokenizers_on_text
from sklearn.metrics import accuracy_score, f1_score
import random
import pandas as pd

In [114]:
# loading training, dev, and test data
train_texts, train_labels = load_split("train")
dev_texts, dev_labels = load_split("dev")
test_texts, test_labels = load_split("test")

print(len(train_texts), len(dev_texts), len(test_texts))

1400 250 250


# Random Sampling of 5 random reviews, comparing different tokenization method

In [115]:
import random

random.seed(100)
# sample 5 random review indices from the training set
sample_indices = random.sample(range(len(train_texts)), 5)

# for each ranomly selected review from the training set, tokenize using all 4 chosen tokenization methods,
# showing the differences between them
for i in range(5):
    text = train_texts[sample_indices[i]]
    
    print("\n------SAMPLE", i + 1, "-------")
    print("ACUTAL TEXT:", text[:400], "...\n")
    
    result = compare_tokenizers_on_text(text)
    
    print("split:")
    print(result["split"][:60])
    print()
    
    print("toktok:")
    print(result["toktok"][:60])
    print()
    
    print("word_tokenize:")
    print(result["word_tokenize"][:60])
    print()
    
    print("wordpunct:")
    print(result["wordpunct"][:60])
    print()


------SAMPLE 1 -------
ACUTAL TEXT: on paper, this movie would sound incredibly boring. the idea of a 75-year-old man traveling the country-side on a riding mower certainly doesn't have much appeal to it, but the real power behind the film is its charm and its intelligence. writers will not find a better study of what makes a movie work than "the straight story." the perfect example of this is a scene in which alvin meets a runaway  ...

split:
['on', 'paper,', 'this', 'movie', 'would', 'sound', 'incredibly', 'boring.', 'the', 'idea', 'of', 'a', '75-year-old', 'man', 'traveling', 'the', 'country-side', 'on', 'a', 'riding', 'mower', 'certainly', "doesn't", 'have', 'much', 'appeal', 'to', 'it,', 'but', 'the', 'real', 'power', 'behind', 'the', 'film', 'is', 'its', 'charm', 'and', 'its', 'intelligence.', 'writers', 'will', 'not', 'find', 'a', 'better', 'study', 'of', 'what', 'makes', 'a', 'movie', 'work', 'than', '"the', 'straight', 'story."', 'the', 'perfect']

toktok:
['on', 'paper', ',

# Helper function for building the model for each combination and running all tests

In [116]:
def run_tests(model_type, vectorizer_type, tokenizer_method):
    vectorizer, clf = build_model(
        model_type=model_type,
        vectorizer_type=vectorizer_type,
        tokenizer_method=tokenizer_method
    )

    train(vectorizer, clf, train_texts, train_labels)
    dev_preds = predict(vectorizer, clf, dev_texts)

    acc = accuracy_score(dev_labels, dev_preds)
    f1 = f1_score(dev_labels, dev_preds)

    return {
        "model": model_type,
        "vectorizer": vectorizer_type,
        "tokenizer": tokenizer_method,
        "dev_accuracy": acc,
        "dev_f1": f1
    }

# Running tests for each combination

In [117]:
model_types = ["logreg", "svm", "nb"]
vectorizer_types = ["tfidf", "count"]
tokenizer_methods = ["split", "toktok", "word_tokenize", "wordpunct"]

results = []

for model_type in model_types:
    for vectorizer_type in vectorizer_types:
        for tokenizer_method in tokenizer_methods:
            result = run_tests(
                model_type=model_type,
                vectorizer_type=vectorizer_type,
                tokenizer_method=tokenizer_method
            )
            results.append(result)
            print(result)

{'model': 'logreg', 'vectorizer': 'tfidf', 'tokenizer': 'split', 'dev_accuracy': 0.848, 'dev_f1': 0.8429752066115702}
{'model': 'logreg', 'vectorizer': 'tfidf', 'tokenizer': 'toktok', 'dev_accuracy': 0.84, 'dev_f1': 0.8412698412698413}
{'model': 'logreg', 'vectorizer': 'tfidf', 'tokenizer': 'word_tokenize', 'dev_accuracy': 0.832, 'dev_f1': 0.832}
{'model': 'logreg', 'vectorizer': 'tfidf', 'tokenizer': 'wordpunct', 'dev_accuracy': 0.852, 'dev_f1': 0.8502024291497976}
{'model': 'logreg', 'vectorizer': 'count', 'tokenizer': 'split', 'dev_accuracy': 0.824, 'dev_f1': 0.824}
{'model': 'logreg', 'vectorizer': 'count', 'tokenizer': 'toktok', 'dev_accuracy': 0.84, 'dev_f1': 0.84375}
{'model': 'logreg', 'vectorizer': 'count', 'tokenizer': 'word_tokenize', 'dev_accuracy': 0.812, 'dev_f1': 0.8127490039840638}
{'model': 'logreg', 'vectorizer': 'count', 'tokenizer': 'wordpunct', 'dev_accuracy': 0.8, 'dev_f1': 0.8015873015873016}
{'model': 'svm', 'vectorizer': 'tfidf', 'tokenizer': 'split', 'dev_accu

# Raw Results

In [118]:
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by="dev_accuracy", ascending=False).reset_index(drop=True)

results_df

,model,vectorizer,tokenizer,dev_accuracy,dev_f1
0,svm,tfidf,word_tokenize,0.868,0.869565
1,svm,tfidf,wordpunct,0.860,0.858300
2,logreg,tfidf,wordpunct,0.852,0.850202
3,nb,tfidf,split,0.852,0.842553
4,logreg,tfidf,split,0.848,0.842975
5,nb,tfidf,wordpunct,0.848,0.838983
6,nb,tfidf,word_tokenize,0.848,0.838983
7,svm,tfidf,split,0.848,0.842975
8,nb,count,word_tokenize,0.840,0.837398
9,logreg,tfidf,toktok,0.840,0.841270


In [119]:
best = results_df.iloc[0]
best

best_vectorizer, best_clf = build_model(
    model_type=best["model"],
    vectorizer_type=best["vectorizer"],
    tokenizer_method=best["tokenizer"],
)

train(best_vectorizer, best_clf, train_texts, train_labels)
test_preds = predict(best_vectorizer, best_clf, test_texts)

print("Test accuracy:", accuracy_score(test_labels, test_preds))
print("Test F1:", f1_score(test_labels, test_preds))

Test accuracy: 0.86
Test F1: 0.8582995951417004


# Error Analysis

In [120]:
wrong_examples = []

for text, true_label, pred_label in zip(test_texts, test_labels, test_preds):
    if true_label != pred_label:
        wrong_examples.append({
            "true_label": true_label,
            "pred_label": pred_label,
            "text_preview": text[:500]
        })

print("Number of wrong examples:", len(wrong_examples))

Number of wrong examples: 35


In [121]:
# getting 10 examples our model got incorrectly for error analysis
for i, ex in enumerate(wrong_examples[:10]):
    print("ERROR:", i + 1)
    print("true label:", ex["true_label"], "predicted label:", ex["pred_label"])
    print(ex["text_preview"])
    print()

ERROR: 1
true label: 1 predicted label: 0
story about three eclipse (maybe even indigo, ha) children beginning their love for murder. oh, and the people who are "hot" on their trail. bloody birthday, a pretty mediocre title for the film, was a nice lil surprise. i was in no way expecting a film that dealt with blood-thirsty psychopath kids. and i may say it's also one of the best flicks i've seen with kids as the villains. by the end of the movie i seriously wanted these kids to die in horrible fashion. it's a really solid 80s horror fl

ERROR: 2
true label: 1 predicted label: 0
this is one of the most satisfying of the book to tv adaptations. the actress who make us believe that a borg could be sexy makes us believe that a spy and traitor can have some redeeming qualities. the tv plot line does not follow cornwell's story exactly but is both exciting and rewarding as a retelling of a darn good yarn. if you have a yen for romance in uniform there is a lot of sexual energy sparking betw

# Visualizaion/Conparison of Results:

In [ ]:
#Visualization/Analysis of results:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import confusion_matrix

viz_df = results_df.rename(columns={"dev_accuracy": "accuracy"})

# --- 1. Heatmap: model × vectorizer (averaged over tokenizers) ---
pivot = viz_df.groupby(["model", "vectorizer"])["accuracy"].mean().unstack()
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.heatmap(
    pivot,
    annot=True, fmt=".3f", cmap="YlGnBu",
    linewidths=0.5, vmin=0.7, vmax=1.0,
    ax=axes[0],
)
axes[0].set_title("Mean Dev Accuracy\n(model × vectorizer, averaged over tokenizers)")
axes[0].set_xlabel("Vectorizer")
axes[0].set_ylabel("Model")

# --- 2. Grouped bar chart: accuracy by model × tokenizer ---
pivot2 = viz_df.groupby(["model", "tokenizer"])["accuracy"].mean().reset_index()
sns.barplot(
    data=pivot2,
    x="tokenizer", y="accuracy", hue="model",
    palette="Set2", ax=axes[1],
)
axes[1].set_title("Mean Dev Accuracy\nby Tokenizer and Model")
axes[1].set_xlabel("Tokenizer")
axes[1].set_ylabel("Accuracy")
axes[1].set_ylim(0.7, 1.0)
axes[1].legend(title="Model")

# --- 3. Confusion matrix for the best model on the dev set ---
best_row = viz_df.loc[viz_df["accuracy"].idxmax()]
print(f"Best config: model={best_row['model']}, vectorizer={best_row['vectorizer']}, tokenizer={best_row['tokenizer']}, acc={best_row['accuracy']:.4f}")

best_dev_preds = predict(best_vectorizer, best_clf, dev_texts)

cm = confusion_matrix(dev_labels, best_dev_preds)
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=["Negative", "Positive"],
    yticklabels=["Negative", "Positive"],
    ax=axes[2],
)
axes[2].set_title(f"Confusion Matrix (dev set)\nBest: {best_row['model']} + {best_row['vectorizer']} + {best_row['tokenizer']}")
axes[2].set_xlabel("Predicted")
axes[2].set_ylabel("Actual")

plt.tight_layout()
plt.show()
